# ADNI-X analysis entry points

All calculations live in the package. The default full run produces diagnosis (CN/MCI/AD) and MoCA-category (29–30, 20–28, 0–19) results. MMSE categories are not used. Set CONFIG to your HPC YAML; each target is independently runnable.


In [ ]:
from pathlib import Path
from adni_x.config import Config
from adni_x import validate, prepare, run, plot
from adni_x.catalogue import catalogue

CONFIG = "../config.local.yaml"


## Display and restyle saved figures
Run the helper cell below once per kernel session. Each analysis is followed by a
separate display cell. After restarting Jupyter, set `RESULTS_RUN` to the existing
run folder and run only the helper and desired display cells.

`show_figures()` displays the newest rendering of each figure. To apply an updated
plotting style to completed analyses, run `refresh_figures("group-matrices")` for
one analysis or `refresh_figures()` for the whole saved run. This reads only saved
numerical bundles and produces timestamped SVG, PDF and 300-dpi PNG files; it does
not repeat validation, preparation, or statistical analyses. Existing renderings
are retained. Restart the kernel after updating the package's plotting code.

Leave `RESULTS_RUN = None` when computing new results to follow the run returned
by the most recent analysis cell.


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Image

# After restarting Jupyter, set this to your existing results/RUN_ID folder.
# Leave as None to use the run returned by the most recent analysis cell.
RESULTS_RUN = None  # e.g. Path("/path/to/results/RUN_ID")


def saved_run():
    run_folder = RESULTS_RUN
    if run_folder is None:
        run_folder = globals().get("result", {}).get("path")
    if run_folder is None:
        raise ValueError("Set RESULTS_RUN to a saved run folder, or run an analysis cell first.")
    return Path(run_folder).expanduser()


def show_figures(analysis):
    """Display the newest saved rendering of each figure, without recomputing."""
    figure_folder = saved_run() / analysis / "figures"
    latest = {}
    for sidecar in sorted(figure_folder.glob("*.json")):
        record = json.loads(sidecar.read_text())
        for name in record.get("files", {}):
            path = figure_folder / name
            if path.suffix != ".png" or not path.is_file():
                continue
            key = record["figure_id"]
            timestamp = record.get("created_at", "")
            if key not in latest or timestamp > latest[key][0]:
                latest[key] = (timestamp, path)
    # The original notebook used a combined coefficient grid.
    if analysis + ".combined" in latest:
        latest.pop(analysis + ".rtr", None)
        latest.pop(analysis + ".sts", None)
    images = [latest[key][1] for key in sorted(latest)]
    if not images:
        images = sorted(figure_folder.glob("*.png"))
    if not images:
        print(f"No saved figures found for {analysis}: {figure_folder}")
    for path in images:
        print(path.name)
        display(Image(filename=str(path)))


def refresh_figures(analysis="all"):
    """Apply the current plotting style to saved results; no input data needed."""
    from adni_x import plot
    root = saved_run()
    return plot("all", root if analysis == "all" else root / analysis)


## Audit inputs
Eligibility requires 150–250 time points and TR 2.875–3.150 seconds, inclusive, before latest-visit selection. Review both diagnosis and MoCA counts.


In [ ]:
audit = validate(CONFIG)
audit.summary()

## Reusable preparation
Preparation is cached. Individual analyses also prepare missing dependencies automatically.

In [ ]:
dataset = prepare(CONFIG)
dataset.path

## Available analyses and figures

In [ ]:
catalogue()

## Run all default analyses for both groupings

Use this cell for a full rerun, or use individual target cells below. Completed targets are reused. MoCA reference analyses use 29–30; MoCA prediction compares 29–30 against 0–19. Missing labels exclude participants only from that grouping, with no older-visit substitution. The two raw TDMI-ratio sensitivity targets require explicit opt-in.


In [ ]:
result = run("all", CONFIG)
result["path"]


## Group atomic matrices and edge contrasts

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("group-matrices", CONFIG)
result["path"]


In [ ]:
show_figures("group-matrices")


## Group atomic matrices and edge contrasts (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("group-matrices-moca", CONFIG)
result["path"]


In [ ]:
show_figures("group-matrices-moca")


## Confound coefficient grids

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("coefficients", CONFIG)
result["path"]


In [ ]:
show_figures("coefficients")


## Confound coefficient grids (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("coefficients-moca", CONFIG)
result["path"]


In [ ]:
show_figures("coefficients-moca")


## Intranetwork atom comparisons and effect sizes

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("network-effects", CONFIG)
result["path"]


In [ ]:
show_figures("network-effects")


## Intranetwork atom comparisons and effect sizes (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("network-effects-moca", CONFIG)
result["path"]


In [ ]:
show_figures("network-effects-moca")


## Regional comparisons, significant counts and network summaries

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("regional-effects", CONFIG)
result["path"]


In [ ]:
show_figures("regional-effects")


## Regional comparisons, significant counts and network summaries (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("regional-effects-moca", CONFIG)
result["path"]


In [ ]:
show_figures("regional-effects-moca")


## Rank hierarchies, profiles and bootstrap correlations

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("hierarchies", CONFIG)
result["path"]


In [ ]:
show_figures("hierarchies")


## Rank hierarchies, profiles and bootstrap correlations (MoCA categories)

Grouping: moca. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("hierarchies-moca", CONFIG)
result["path"]


In [ ]:
show_figures("hierarchies-moca")


## Four taxa by DX

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("taxa-four-dx", CONFIG)
result["path"]


In [ ]:
show_figures("taxa-four-dx")


## Four taxa by MoCA (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("taxa-four-moca", CONFIG)
result["path"]


In [ ]:
show_figures("taxa-four-moca")


## Six taxa by DX

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("taxa-six-dx", CONFIG)
result["path"]


In [ ]:
show_figures("taxa-six-dx")


## Six taxa by MoCA (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("taxa-six-moca", CONFIG)
result["path"]


In [ ]:
show_figures("taxa-six-moca")


## CN-standardised regional comparisons, PCA and brain exports

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("cn-reference", CONFIG)
result["path"]


In [ ]:
show_figures("cn-reference")


## MoCA 29–30-standardised regional comparisons, PCA and brain exports (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("moca-reference", CONFIG)
result["path"]


In [ ]:
show_figures("moca-reference")


## Held-out CN/AD LDA and labelled projections

Grouping: diagnosis. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("lda", CONFIG)
result["path"]


In [ ]:
show_figures("lda")


## Held-out MoCA 29–30/0–19 LDA and labelled projections (MoCA categories)

Grouping: moca. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("lda-moca", CONFIG)
result["path"]


In [ ]:
show_figures("lda-moca")


## Held-out participant hierarchy correlations

Grouping: diagnosis. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("heldout-hierarchy", CONFIG)
result["path"]


In [ ]:
show_figures("heldout-hierarchy")


## Held-out participant hierarchy correlations (MoCA categories)

Grouping: moca. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("heldout-hierarchy-moca", CONFIG)
result["path"]


In [ ]:
show_figures("heldout-hierarchy-moca")


## Alteration correlations and exploratory network permutations

Grouping: diagnosis. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("alterations", CONFIG)
result["path"]


In [ ]:
show_figures("alterations")


## Alteration correlations and exploratory network permutations (MoCA categories)

Grouping: moca. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("alterations-moca", CONFIG)
result["path"]


In [ ]:
show_figures("alterations-moca")


## Raw all-visit atom and TDMI trajectories

Grouping: diagnosis. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("longitudinal", CONFIG)
result["path"]


In [ ]:
show_figures("longitudinal")


## Raw all-visit atom and TDMI trajectories (MoCA categories)

Grouping: moca. Exploratory analysis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("longitudinal-moca", CONFIG)
result["path"]


In [ ]:
show_figures("longitudinal-moca")


## Optional raw atom/TDMI longitudinal sensitivity

Grouping: diagnosis. Exploratory analysis. Running this cell explicitly opts into TDMI-ratio sensitivity. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("longitudinal-tdmi-ratio", CONFIG)
result["path"]


In [ ]:
show_figures("longitudinal-tdmi-ratio")


## Optional raw atom/TDMI longitudinal sensitivity (MoCA categories)

Grouping: moca. Exploratory analysis. Running this cell explicitly opts into TDMI-ratio sensitivity. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("longitudinal-tdmi-ratio-moca", CONFIG)
result["path"]


In [ ]:
show_figures("longitudinal-tdmi-ratio-moca")


## Raw global TDMI, diagnosis comparisons and length audit

Grouping: diagnosis. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("tdmi", CONFIG)
result["path"]


In [ ]:
show_figures("tdmi")


## Raw global TDMI, MoCA comparisons and length audit (MoCA categories)

Grouping: moca. Results, cohort tables, figures and provenance are saved together.


In [ ]:
result = run("tdmi-moca", CONFIG)
result["path"]


In [ ]:
show_figures("tdmi-moca")


## Replot one figure without inputs
Replace the path with an existing saved analysis bundle. The figure timestamp records this rendering, not the original statistical computation.

In [ ]:
from adni_x import plot
# plot("group-matrices.rtr", "/path/to/saved/run/group-matrices")